# Объединения, пропуски и дубликаты

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Проверять объединения таблиц и обрабатывать пропуски без незаметного изменения набора объектов.

## Таблица для примеров

Создадим четыре вымышленных измерения. Значение `None` обозначает пропуск массы. Идентификатор связывает данные об одном объекте между таблицами.

In [ ]:
import pandas as pd
frame = pd.DataFrame({
    "id": ["a", "b", "c", "d"],
    "span_cm": [12., 22., 14., 24.],
    "mass_g": [8., 10., None, 12.],
    "flight": ["короткий", "дальний", "короткий", "дальний"],
})


## Пропуски и дубликаты

`is_unique` истинно, если в столбце нет повторов. `duplicated()` отмечает повторяющиеся строки, начиная со второго вхождения; сумма маски даёт их число. `fillna(value)` возвращает столбец, в котором пропуски заменены на `value`.

In [ ]:
assert frame["id"].is_unique
assert frame.duplicated().sum() == 0
filled = frame.copy()
filled["mass_g"] = filled["mass_g"].fillna(10.)
assert filled["mass_g"].isna().sum() == 0
assert frame["mass_g"].isna().sum() == 1


Число 10 здесь — заранее выбранное учебное заполнение, не результат анализа проверочных данных. В модели медиану или другое правило подбирают на обучающей части. До удаления повторов выясните, что повторилось: вся строка, идентификатор или законное повторное измерение. Удаление одинаковых строк не гарантирует уникальности `id`.

## Разобранный пример: объединение с контролем

Добавим данные о бумаге. `merge` объединяет по ключу, `how="left"` сохраняет все строки левой таблицы. `validate="one_to_one"` требует уникальных ключей с обеих сторон; `indicator=True` добавляет столбец `_merge` со сведениями о совпадении ключа. Значение `"both"` означает, что ключ найден в обеих таблицах; `.all()` проверяет истинность условия для каждой строки. Поле `paper_gsm` — масса бумаги площадью один квадратный метр, в г/м².

In [ ]:
paper = pd.DataFrame({"id": ["a", "b", "c", "d"], "paper_gsm": [80, 90, 80, 90]})
joined = frame.merge(paper, on="id", how="left", validate="one_to_one", indicator=True)
assert len(joined) == len(frame)
assert (joined["_merge"] == "both").all()


Если справа два одинаковых ключа, строки могут размножиться. Если ключ отсутствует, появится пропуск. Проверяйте и число строк, и то, для каких ключей нашлось соответствие. Для справочника с повторяющимися ключами слева и уникальными справа подходит `many_to_one`. Не допускайте пустых ключей: их объединение может иметь неожиданный смысл.

## Почему число строк меняется

Если ключ встречается слева дважды, а справа трижды, каждая из двух строк сочетается с каждой из трёх: получится шесть строк. Это может быть правильной задачей перебора сочетаний, но для справочника бумаги обычно означает ошибку ключа.

При `how="left"` отсутствие правого ключа не удаляет левую строку, а создаёт пропуск правых полей. Поэтому сохранение числа строк ещё не доказывает успешного объединения: проверьте `_merge` и заполненность нужных признаков.

## Пропущенное соответствие тоже нужно обнаружить

Возьмём справочник только для объекта a. Левое объединение сохранит четыре объекта, но для b, c и d поля бумаги останутся пустыми.

In [ ]:
incomplete = paper.iloc[:1]
checked = frame.merge(incomplete, on="id", how="left",
                      validate="one_to_one", indicator=True)
assert len(checked) == 4
assert checked.loc[checked["_merge"] == "left_only", "id"].tolist() == ["b", "c", "d"]


Значение `left_only` означает «найден только слева». Условие `validate="one_to_one"` проверяет уникальность ключей, но не требует, чтобы для каждого левого ключа нашлось правое соответствие. Поэтому нужны обе проверки.

## Задачи

### 1. Базовая

Что проверяет is_unique у столбца id?

<details>
<summary>Подсказка</summary>

Проверяется повторение значений.

</details>



### 2. Базовая

Удаляет ли left merge левую строку без правого соответствия?

<details>
<summary>Подсказка</summary>

Левая таблица сохраняется.

</details>



### 3. Прикладная

Найдите id с отсутствующей массой.

<details>
<summary>Подсказка</summary>

Используйте isna и loc.

</details>



### 4. Прикладная

Заполните пропуск массой 10 в независимой копии.

<details>
<summary>Подсказка</summary>

Не меняйте frame.

</details>



### 5. Олимпиадная

Слева ключ a повторён дважды, справа трижды. Сколько совпавших строк получится?

<details>
<summary>Подсказка</summary>

Каждая левая строка сочетается с каждой правой.

</details>



### 6. Олимпиадная

Почему удаление одинаковых строк не гарантирует уникальность id?

<details>
<summary>Подсказка</summary>

Поля массы могут различаться.

</details>



## Итоги

Вы умеете заполнять пропуски в копии и проверять уникальность и покрытие ключей объединения.

## Самопроверка

Почему одного равенства числа строк до и после `merge` недостаточно? Что отличает `one_to_one` от `many_to_one` и о чём сообщает `left_only`?

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

Отсутствие повторяющихся идентификаторов.

### Задача 2

Нет, правые поля становятся пропущенными.

### Задача 3

In [ ]:
assert frame.loc[frame["mass_g"].isna(), "id"].tolist() == ["c"]


### Задача 4

In [ ]:
working = frame.copy()
working["mass_g"] = working["mass_g"].fillna(10.)
assert working["mass_g"].isna().sum() == 0
assert frame["mass_g"].isna().sum() == 1


### Задача 5

Шесть.

### Задача 6

Строки с одним id и разными значениями не являются полными дубликатами. Проверять нужно именно ключ и смысл повторных измерений.